# 05 · Transformación y carga local en PostgreSQL

## Contexto
Quinta fase del pipeline ETL. Transforma los datos brutos extraídos 
de RAWG y los carga en la base de datos PostgreSQL local.

## Objetivo
- Transformar los datos JSON en 12 DataFrames estructurados
- Cargar los DataFrames en PostgreSQL local usando SQLAlchemy

## Requisitos
- Base de datos `rawg_db_local` creada (ver notebook `04`)
- PostgreSQL corriendo en localhost:5432
- Credenciales configuradas como `YOUR_LOCAL_PASSWORD`
- Archivo `extraccion_historica.json` disponible en `data/`

## Orden de carga
1. Dimensiones: `esrb_ratings` · `platforms` · `genres` · `stores` · `tags`
2. Tabla principal: `games`
3. Tablas dependientes: `games_status` · `ratings_distribution`
4. Tablas N:M: `game_platforms` · `game_genres` · `game_stores` · `game_tags`

## Input
`data/extraccion_historica.json`

## Output
Base de datos `rawg_db_local` poblada con ~20.000 videojuegos

In [ ]:
import json

import pandas as pd
import sqlalchemy
from sqlalchemy import create_engine, text


from datetime import datetime
from io import StringIO

import boto3
import time
from botocore.exceptions import ClientError

import sys

RUTA_SCRIPTS_ETL = r'../etl'

if RUTA_SCRIPTS_ETL not in sys.path:
    sys.path.insert(0, RUTA_SCRIPTS_ETL)

In [ ]:
# Verificar versiones
print(f"pandas=={pd.__version__}")
print(f"sqlalchemy=={sqlalchemy.__version__}")

In [ ]:
# === CARGAR DATOS RAWG ===
# ============================

import json
import transformation_data_rawg

print("Cargando datos brutos...\n")
with open('../data/extraccion_historica.json', 'r') as f:
    datos = json.load(f)

print(f"Total de juegos cargados: {len(datos)}")


In [ ]:
# === TRANSFORMAR ===
# ==================

print("Ejecutando transformación...")

from transform_rawg import transf_rawg_data

dataframes = transf_rawg_data(datos, verbose = True)

# Extraer DataFrames
esrb_ratings = dataframes['esrb_ratings']
platforms = dataframes['platforms']
genres = dataframes['genres']
stores = dataframes['stores']
tags = dataframes['tags']
games = dataframes['games']
games_status = dataframes['games_status']
ratings_distribution = dataframes['ratings_distribution']
game_platforms = dataframes['game_platforms']
game_genres = dataframes['game_genres']
game_stores = dataframes['game_stores']
game_tags = dataframes['game_tags']

print("DataFrames listos para cargar\n")

In [ ]:
# === CONECTAR A BASE DE DATOS: CREAR ENGINE DE CONEXIÓN ===

host = "localhost"
user = "postgres"
password = "YOUR_LOCAL_PASSWORD"
database = "rawg_db_local"
port = 5432

# Crear la conexión
engine = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}")

# Abrir la conexión
connection = engine.connect()

# Cerrar la conexión
connection.close()


In [ ]:
# === VERIFICAR CONEXIÓN ===
try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT version();"))
        print("Conexión exitosa a PostgreSQL")
        print("-" * 60)
except Exception as e:
    print(f"Error de conexión: {e}")
    raise


In [ ]:

# === FUNCIÓN DE CARGA MASIVA ===

def cargar_dataframe(df, tabla_name, engine, if_exists='append'):
    """
    Carga un DataFrame en PostgreSQL usando pandas.to_sql()
        df: DataFrame a cargar
        tabla_name: nombre de la tabla
        engine: SQLAlchemy engine
        if_exists: 'append' (agregar)
    """
    if df.empty:
        print(f"DataFrame vacío para '{tabla_name}', saltando...")
        return
    
    try:
        df.to_sql(
            name = tabla_name,
            con = engine,
            schema = 'rawg',
            if_exists = if_exists,
            index = False,
            method= 'multi',
            chunksize = 1000
        )
        print(f"{len(df):,} filas cargadas en 'rawg.{tabla_name}'")
        
    except Exception as e:
        print(f"Error en 'rawg.{tabla_name}': {e}")



In [ ]:
# === PROCESO DE CARGA ===

print("\nINICIANDO CARGA MASIVA CON SQLALCHEMY + PANDAS")


# DIMENSIONES / CATÁLOGOS
print("\nCargando dimensiones y catálogos...")

cargar_dataframe(esrb_ratings, 'esrb_ratings', engine)
cargar_dataframe(platforms, 'platforms', engine)
cargar_dataframe(genres, 'genres', engine)
cargar_dataframe(stores, 'stores', engine)
cargar_dataframe(tags, 'tags', engine)

# TABLA PRINCIPAL
print("\nCargando tabla principal de juegos...")

cargar_dataframe(games, 'games', engine)

# TABLAS DEPENDIENTES
print("\nCargando tablas dependientes de games...")

cargar_dataframe(games_status, 'games_status', engine)
cargar_dataframe(ratings_distribution, 'ratings_distribution', engine)

# TABLAS DE RELACIÓN N:M 
print("\nCargando tablas de relación (N:M)...")
print("-" * 60)

cargar_dataframe(game_platforms, 'game_platforms', engine)
cargar_dataframe(game_genres, 'game_genres', engine)
cargar_dataframe(game_stores, 'game_stores', engine)
cargar_dataframe(game_tags, 'game_tags', engine)


# === RESUMEN FINAL ===
print("\nCarga completada con éxito en la base de datos local.")
print("="*60)